# Data analysis

Every number in `docs/data_traps.md` should come from a cell in this notebook.

In [1]:
from pathlib import Path
import sys, json, re, csv, collections, math, zipfile, hashlib
import xml.etree.ElementTree as ET

ROOT = Path.cwd() if (Path.cwd() / "analysis").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from analysis.load import (DATA, load_alerts, load_sample_alerts, load_log,
                           load_inventory, load_intel, load_context, log_sources)
from analysis.helpers import (leaves, raw_records, encoded, exact_mask,
                              entity_values, change_times, change_window,
                              approval_refs, RUNBOOK_SOURCES)

def show(title, value):
    print(title)
    if isinstance(value, pd.Series):
        value = value.rename_axis("value").reset_index(name="count")
    if isinstance(value, pd.DataFrame):
        print(value.to_string(index=False, max_rows=50, max_colwidth=90))
    else:
        print(value)
    print()

# Tables below contain IDs/counts/controlled descriptions, never raw attacker prose.
# All joins and scans are local; no LLM, external indicator lookup or network I/O.


## 1. Overview

How many alerts are there by rule, product, severity and day? Which rules dominate? Are there any answer labels?

Affects: data/eval

In [2]:
alerts = load_alerts()
sample = load_sample_alerts()
logs = {source: load_log(source) for source in log_sources()}
inventory_names = sorted(p.stem for p in (DATA / "02_inventario").glob("*.csv"))
inventory = {name: load_inventory(name) for name in inventory_names}
assets, users = inventory["activos"], inventory["usuarios"]
changes, hr = inventory["calendario_cambios"], inventory["eventos_rrhh"]
services = inventory["servicios_negocio"]
catalog = load_context("catalogo_casos_de_uso")
intel = {name: load_intel(name) for name in ["ti_feed", "geoip_asn", "sandbox_informes"]}
show("Dataset counts", pd.DataFrame([
    ["alerts", len(alerts)], ["sample", len(sample)],
    ["log sources", len(logs)], ["log events", sum(map(len, logs.values()))],
    ["assets", len(assets)], ["users", len(users)], ["catalog rules", len(catalog)]
], columns=["table", "rows"]))
rule_counts = alerts.groupby(["rule_id", "source_product", "severity"]).size().reset_index(name="alerts")
show("All observed rules (rule severity, not final severity)", rule_counts.sort_values("alerts", ascending=False))
show("Products", alerts.source_product.value_counts())
show("Rule severities", alerts.severity.value_counts())
show("Alerts by UTC date", alerts.groupby(alerts.created_at_utc.dt.date).size())
label_fields = sorted({k for frame in [alerts, sample, *logs.values(), *inventory.values()]
                       for k in frame if re.search(r"ground.?truth|label|incident_id|verdict", k, re.I)})
show("Potential label fields (detections are not answer labels)", label_fields)
show("Catalog rules absent from alerts", sorted(set(catalog.rule_id) - set(alerts.rule_id)))
show("Exact alert/sample ID checks", {
    "duplicate_alert_ids": int(alerts.alert_id.duplicated().sum()),
    "duplicate_sample_ids": int(sample.alert_id.duplicated().sum()),
    "sample_ids_outside_full_set": len(set(sample.alert_id)-set(alerts.alert_id)),
})
full_by_id = {r["alert_id"]: r for r in raw_records(alerts)}
sample_differences = sum(r != full_by_id.get(r["alert_id"]) for r in raw_records(sample))
show("Sample row differences", sample_differences)


Dataset counts
        table   rows
       alerts  11061
       sample    150
  log sources     18
   log events 814471
       assets   1185
        users   1224
catalog rules     44

All observed rules (rule severity, not final severity)
   rule_id source_product severity  alerts
UC-WEB-001            WAF      low    3406
UC-EML-001       EMAIL_GW      low    2295
UC-EDR-002            EDR   medium    1349
UC-IDP-001            IDP   medium     901
UC-NET-001          PROXY   medium     746
UC-NET-005            DNS      low     561
UC-EDR-001            EDR     high     532
 UC-AI-001     AI_GATEWAY   medium     415
UC-EDR-003            EDR   medium     411
UC-NET-006            DNS   medium      91
UC-WEB-002            WAF     high      77
UC-IDP-002            IDP     high      62
UC-DLP-001            DLP     high      59
UC-CLD-003          CLOUD   medium      50
UC-NET-003           NGFW   medium      20
UC-WEB-003           ECOM     high      12
UC-DEV-001           CICD   me

**Observed:** Counts above are recomputed from files; detection verdicts in email, sandbox and AI logs describe particular controls, not ground truth for the alert. There is no supplied alert-level incident/verdict answer column. The README's opening source count and the deck's runbook count should not be used as schema contracts.

**Design interpretation (data/eval):** Keep raw rule severity and control verdicts separate from analyst labels. Build an independently reviewed evaluation set; do not treat the development sample as a scored test set.

## 2. Noise

For the loudest rules, group alerts by their key evidence fields. How many distinct patterns are there, and which alerts stand out?

Affects: agent/prompt, data/eval

In [3]:
# Selected evidence keys deliberately exclude message/event IDs and unique timestamps.
pattern_keys = {
 "UC-WEB-001": ["rule_id", "action"],
 "UC-EML-001": ["sender", "subject"],
 "UC-EDR-002": ["parent", "user", "decoded_preview"],
 "UC-IDP-001": ["device", "note"],
 "UC-NET-001": ["domain", "user_agent", "hosts_type"],
 "UC-NET-005": ["parent_domain"],
 "UC-EDR-001": ["process_tree", "signer", "edr_action"],
 "UC-AI-001": ["guardrail_category", "action"],
}
pattern_summary, pattern_examples = [], []
for rule in alerts.rule_id.value_counts().head(8).index:
    part = alerts[alerts.rule_id.eq(rule)].copy()
    keys = pattern_keys[rule]
    part["pattern"] = part.evidence.map(lambda e: encoded({k:e.get(k) for k in keys}))
    frequencies = part.pattern.value_counts()
    sizes = part.pattern.map(frequencies)
    pattern_summary.append([rule, len(part), len(frequencies), int(frequencies.iloc[0]),
                            int(sizes.eq(1).sum()), ", ".join(keys)])
    # Emit only IDs, never the pattern text, which may contain planted instructions.
    for _, r in part.assign(pattern_size=sizes).sort_values(["pattern_size", "alert_id"]).head(2).iterrows():
        pattern_examples.append([rule, r.alert_id, int(r.pattern_size)])
show("Largest rules: patterns under explicitly chosen fields", pd.DataFrame(pattern_summary,
    columns=["rule", "alerts", "patterns", "largest_pattern", "singleton_alerts", "group_fields"]))
show("Smallest-pattern leads (two per rule; rarity is not a verdict)", pd.DataFrame(
    pattern_examples, columns=["rule", "alert_id", "pattern_alerts"]))
show("Overall exact duplicate alert records (ignoring ID and creation time)",
     pd.Series([encoded({k:v for k,v in r.items() if k not in {"alert_id", "created_at"}})
                for r in raw_records(alerts)]).duplicated().sum())


Largest rules: patterns under explicitly chosen fields
      rule  alerts  patterns  largest_pattern  singleton_alerts                     group_fields
UC-WEB-001    3406         8              731                 0                  rule_id, action
UC-EML-001    2295         8              350                 0                  sender, subject
UC-EDR-002    1349         1             1349                 0    parent, user, decoded_preview
UC-IDP-001     901         3              899                 2                     device, note
UC-NET-001     746         7              161                 2   domain, user_agent, hosts_type
UC-NET-005     561         6              101                 0                    parent_domain
UC-EDR-001     532         2              531                 1 process_tree, signer, edr_action
 UC-AI-001     415         2              413                 0       guardrail_category, action

Smallest-pattern leads (two per rule; rarity is not a verdict)
      ru

Overall exact duplicate alert records (ignoring ID and creation time)
0



**Observed:** Pattern counts depend on the displayed grouping fields. A changing subject or command can create a singleton; many different endpoints can share one pattern.

**Design interpretation (agent/prompt, data/eval):** Retrieve representative evidence and count repetitions, retaining exceptions. These groups are compression candidates and investigation leads. They do not justify closing every matching alert or labelling rare alerts malicious.

## 3. Identities

How do people, machines and IPs appear in each log? Does the mock API's q search ever return the wrong rows, or miss rows?

Affects: data/search-logs

In [4]:
identity_columns = {s:[c for c in d if re.search(
    r"user|host|ip$|IPAddress|Identity|recipient|sender|customer|session|actor|subject|target|runner|^dc$", c)]
    for s,d in logs.items()}
show("Identity-bearing fields by source (nested fields still need parsing)", pd.DataFrame(
    [[s, ", ".join(c)] for s,c in identity_columns.items()], columns=["source","fields"]))
prefix_pairs = [(a,b) for a in users.username for b in users.username if a != b and b.startswith(a)]
show("Directory collisions", {"username_prefix_pairs":len(prefix_pairs),
    "nonunique_display_names": int(users.display_name.duplicated(keep=False).sum()),
    "duplicate_upns":int(users.upn.str.casefold().duplicated().sum()),
    "duplicate_asset_ips":int(assets.ip.duplicated().sum())})
# Reproduce q exactly as implemented, without its limit, then compare to exact aliases.
query = "laura.ferrer"
u = users[users.username.eq(query)].iloc[0]
d = logs["idp_signins"]
q_mask = pd.Series([query in encoded(r).lower() for r in raw_records(d)], index=d.index)
e_mask = exact_mask(d, [u.username, u.upn])
show("Username substring vs exact directory aliases", pd.DataFrame([
    [query, int(q_mask.sum()), int(e_mask.sum()), int((q_mask & ~e_mask).sum()), int((e_mask & ~q_mask).sum())]
], columns=["q","substring_rows","exact_rows","extra_rows","missed_rows"]))
# A machine name is absent in firewall records, which identify it by IP.
asset = assets[assets.hostname.eq("JUMP01")].iloc[0]
f = logs["firewall"]
q_host = pd.Series([asset.hostname.lower() in encoded(r).lower() for r in raw_records(f)], index=f.index)
ip_match = f.src_ip.eq(asset.ip) | f.dst_ip.eq(asset.ip)
show("Hostname lookup against IP-only firewall", {"hostname":"JUMP01", "q_rows":int(q_host.sum()),
     "exact_ip_rows":int(ip_match.sum()), "missed_by_hostname":int((ip_match & ~q_host).sum())})
show("Coverage of raw identity fields", pd.DataFrame([
    [s, sum(logs[s][c].isna().sum() + logs[s][c].eq("").sum() for c in cols if logs[s][c].map(lambda x:isinstance(x,str) or x is None).all())]
    for s,cols in identity_columns.items()], columns=["source", "empty_scalar_identity_cells"]))


Identity-bearing fields by source (nested fields still need parsing)
      source                                     fields
   ad_events              dc, subject, target, src_host
  ai_gateway            session_id, customer_id, src_ip
 cicd_builds                                     runner
  cloudtrail   sourceIPAddress, userAgent, userIdentity
         dlp                             user, hostname
         dns                           src_ip, hostname
ecom_account                        customer_id, src_ip
   ecom_auth            customer_id, src_ip, user_agent
  edr_events                             hostname, user
       email     sender, recipients, subject, sender_ip
email_clicks                               user, src_ip
    firewall                             src_ip, dst_ip
   idp_audit                  actor_upn, target, src_ip
 idp_signins   user_upn, src_ip, user_agent, session_id
       proxy src_ip, hostname, user, dst_ip, user_agent
   sql_audit                       

Directory collisions
{'username_prefix_pairs': 149, 'nonunique_display_names': 281, 'duplicate_upns': 0, 'duplicate_asset_ips': 0}



Username substring vs exact directory aliases
           q  substring_rows  exact_rows  extra_rows  missed_rows
laura.ferrer             153          73          80            0



Hostname lookup against IP-only firewall
{'hostname': 'JUMP01', 'q_rows': 0, 'exact_ip_rows': 220, 'missed_by_hostname': 220}



Coverage of raw identity fields
      source  empty_scalar_identity_cells
   ad_events                            0
  ai_gateway                            0
 cicd_builds                            0
  cloudtrail                            0
         dlp                            0
         dns                            0
ecom_account                            0
   ecom_auth                            0
  edr_events                            7
       email                            0
email_clicks                            0
    firewall                            0
   idp_audit                            0
 idp_signins                         3801
       proxy                            0
   sql_audit                            0
         vpn                          167
         waf                            0



**Observed:** The concrete username query can mix separate identities; hostname search cannot find IP-only firewall rows. Neither result is an API outage. `userIdentity`, `details`, recipient arrays and customer/session identifiers require source-specific extraction.

**Design interpretation (data/search-logs):** Resolve exact UPN/username/hostname/IP aliases before searching. Filter returned rows by parsed field equality; do not use a display name as a join key. Searching a hostname alone is insufficient. Substring counts here precede the API's limit; section 12 checks the actual route.

## 4. Links

Is event_id unique across logs? Do related_events always resolve? Are logs sorted by time, and what does the API's limit cut off?

Affects: data/search-logs, agent/triage

In [5]:
event_index = collections.defaultdict(list)
link_stats = []
for source, d in logs.items():
    for i, event_id in d.event_id.items():
        event_index[event_id].append((source, i))
    link_stats.append([source,len(d),int(d.event_id.duplicated().sum()),
        int(d.timestamp_utc.isna().sum()), int(d.timestamp_utc.diff().lt(pd.Timedelta(0)).sum())])
refs = [(a.alert_id, event_id) for a in alerts.itertuples() for event_id in a.related_events]
missing_refs = [(aid,eid) for aid,eid in refs if eid not in event_index]
ambiguous_refs = [(aid,eid) for aid,eid in refs if len(event_index.get(eid, [])) > 1]
show("Event identity and chronological checks", pd.DataFrame(link_stats,
    columns=["source","rows","duplicate_ids_within","invalid_time","backward_steps"]))
show("Link resolution", {"reference_occurrences":len(refs), "unique_referenced_ids":len(set(e for _,e in refs)),
    "cross_source_or_within_duplicate_ids":sum(len(v)>1 for v in event_index.values()),
    "missing_reference_occurrences":len(missing_refs), "ambiguous_reference_occurrences":len(ambiguous_refs),
    "unreferenced_log_events":sum(map(len,logs.values()))-len({e for _,e in refs if e in event_index})})
if missing_refs:
    show("Missing reference examples (IDs only)",pd.DataFrame(missing_refs[:8],columns=["alert_id","event_id"]))
center = pd.Timestamp("2026-10-07T08:35:15Z")
cutoff_rows=[]
for source in ["proxy","idp_signins","idp_audit","email"]:
    d=logs[source]
    part=d[d.timestamp_utc.between(center-pd.Timedelta(hours=24),center+pd.Timedelta(hours=24))]
    cutoff_rows.append([source,len(part),min(200,len(part)),max(0,len(part)-200),
        part.timestamp_utc.iloc[199].isoformat() if len(part)>=200 else "not reached"])
show("API first-200 cutoff in one unfiltered ±24 h window",pd.DataFrame(cutoff_rows,
    columns=["source","matching_rows","returned","omitted","last_returned_utc"]))
show("Alert ingestion lag vs latest referenced event (minutes)",pd.Series([
    (a.created_at_utc-max(logs[s].loc[i,"timestamp_utc"] for e in a.related_events for s,i in event_index.get(e,[]))).total_seconds()/60
    for a in alerts.itertuples() if any(e in event_index for e in a.related_events)
]).describe(percentiles=[.5,.95]).round(2).to_frame("minutes").reset_index(names="stat"))

lag_rows=[]
for a in alerts.itertuples():
    times=[logs[s].loc[i,"timestamp_utc"] for e in a.related_events for s,i in event_index.get(e,[])]
    if times and max(times)>a.created_at_utc:
        lag_rows.append([a.alert_id,round((max(times)-a.created_at_utc).total_seconds()/60,2)])
show("Alerts linking events later than alert creation",{"alerts":len(lag_rows),
    "alerts_without_related_events":int(alerts.related_events.map(len).eq(0).sum())})
show("Largest future-link leads (do not silently rewrite timestamps)",pd.DataFrame(
    sorted(lag_rows,key=lambda x:-x[1])[:3],columns=["alert_id","minutes_after_creation"]))


Event identity and chronological checks
      source   rows  duplicate_ids_within  invalid_time  backward_steps
   ad_events    526                     0             0               0
  ai_gateway   8817                     0             0               0
 cicd_builds    559                     0             0               0
  cloudtrail   6496                     0             0               0
         dlp   1042                     0             0               0
         dns 178221                     0             0               0
ecom_account   7661                     0             0               0
   ecom_auth  44064                     0             0               0
  edr_events 112492                     0             0               0
       email  56797                     0             0               0
email_clicks  16249                     0             0               0
    firewall  24648                     0             0               0
   idp_audit    762     

Alert ingestion lag vs latest referenced event (minutes)
 stat  minutes
count 11060.00
 mean    14.33
  std    36.65
  min  -174.65
  50%     3.10
  95%    95.33
  max   163.10



Alerts linking events later than alert creation
{'alerts': 336, 'alerts_without_related_events': 1}

Largest future-link leads (do not silently rewrite timestamps)
           alert_id  minutes_after_creation
ALR-20261006-000586                  174.65
ALR-20261023-007315                  174.65
ALR-20261026-008540                  174.65



**Observed:** The tables check every event ID and every explicit alert reference. File order is measured rather than assumed. `related_events` is a selected subset of evidence, not a complete incident membership list. In the displayed windows, taking the first 200 omits later evidence.

**Design interpretation (data/search-logs, agent/triage):** Store source plus event ID as provenance. Check `truncated`, split saturated time windows and deduplicate boundary results; the log endpoint has no offset. Correlate using event timestamps, preserving alert creation time separately.

## 5. Context size

How many rows and KB do the logs from 24 hours either side give, per source? Which sources matter for each rule or runbook? How much must be trimmed for a prompt under 8,000 tokens?

Affects: data/context, agent/prompt

In [6]:
# Exact serialized UTF-8 sizes for whole event records as the API would supply.
# Includes field names/JSON punctuation. No tokenizer or model is called.
window_sizes, size_arrays = [], {}
centers=alerts.created_at_utc.sort_values()
for source,d in logs.items():
    ordered=d.sort_values("timestamp_utc")
    times=pd.DatetimeIndex(ordered.timestamp_utc)
    lengths=pd.Series([len(encoded(r).encode("utf-8")) for r in raw_records(ordered)])
    prefix=pd.concat([pd.Series([0]), lengths.cumsum()],ignore_index=True).to_numpy()
    left=times.searchsorted(centers-pd.Timedelta(hours=24),side="left")
    right=times.searchsorted(centers+pd.Timedelta(hours=24),side="right")
    counts=pd.Series(right-left)
    kb=pd.Series((prefix[right]-prefix[left])/1024)
    size_arrays[source]=(counts,kb)
    window_sizes.append([source,int(counts.median()),int(counts.quantile(.95)),round(kb.median(),1),
                         round(kb.quantile(.95),1),round(kb.max(),1)])
show("All-alert ±24 h windows, before entity filtering",pd.DataFrame(window_sizes,
    columns=["source","rows_p50","rows_p95","KB_p50","KB_p95","KB_max"]))
runbook_budget=[]
for rb,sources in RUNBOOK_SOURCES.items():
    candidates=alerts[alerts.runbook.eq(rb)].sort_values("created_at_utc")
    if candidates.empty:continue
    a=candidates.iloc[len(candidates)//2]
    values=entity_values(a)
    lo,hi=a.created_at_utc-pd.Timedelta(hours=24),a.created_at_utc+pd.Timedelta(hours=24)
    count,total_bytes=0,0
    for source in sources:
        d=logs[source]
        part=d[d.timestamp_utc.between(lo,hi)]
        part=part[exact_mask(part,values)]
        count+=len(part)
        total_bytes+=sum(len(encoded(r).encode("utf-8")) for r in raw_records(part))
    # 6000 tokens reserved for evidence, 2000 for policy/runbook/output/instructions.
    # bytes/4 is only a rough planning estimate; byte count is a conservative ceiling.
    runbook_budget.append([rb,a.alert_id,", ".join(sources),count,round(total_bytes/1024,1),
        math.ceil(total_bytes/4),round(max(0,1-24000/max(1,total_bytes))*100,1),
        round(max(0,1-6000/max(1,total_bytes))*100,1)])
show("One representative alert per runbook, exact entities plus ±24 h",pd.DataFrame(runbook_budget,
    columns=["runbook","alert_id","sources","rows","KB","est_tokens_bytes_div_4",
             "trim_pct_est_6000_tokens","trim_pct_6000_byte_ceiling"]))

total_rows=sum((v[0] for v in size_arrays.values()))
total_kb=sum((v[1] for v in size_arrays.values()))
show("Combined all-source ±24 h context over all alert centres",{
    "median_rows":int(total_rows.median()),"p95_rows":int(total_rows.quantile(.95)),
    "median_KB":round(total_kb.median(),1),"p95_KB":round(total_kb.quantile(.95),1)})


All-alert ±24 h windows, before entity filtering
      source  rows_p50  rows_p95  KB_p50  KB_p95  KB_max
   ad_events        37        46     8.8    10.9    11.5
  ai_gateway       629       639   301.5   305.9   313.8
 cicd_builds        44        72     9.7    15.8    17.3
  cloudtrail       365      1763   153.3   848.6   857.1
         dlp        75        80    22.9    24.6    25.9
         dns     16809     17517  3437.8  3612.2  3653.8
ecom_account       513       658   115.4   148.3   161.3
   ecom_auth      2652      4749   982.3  1817.2  2657.5
  edr_events     10796     10999  5485.0  5593.0  5640.7
       email      5371      5527  2555.1  2640.6  2678.9
email_clicks      1519      1613   416.1   441.4   448.2
    firewall      1906      2281   541.8   644.5   652.7
   idp_audit        62       101    18.0    29.7    31.8
 idp_signins      7298      7486  4309.0  4426.7  4493.1
       proxy     18538     18876  9172.7  9346.8  9761.4
   sql_audit       280       291    63.

One representative alert per runbook, exact entities plus ±24 h
runbook            alert_id                                                        sources  rows     KB  est_tokens_bytes_div_4  trim_pct_est_6000_tokens  trim_pct_6000_byte_ceiling
  RB-01 ALR-20261019-005837 email, email_clicks, proxy, edr_events, idp_signins, idp_audit    89   45.2                   11569                      48.1                        87.0
  RB-02 ALR-20261017-005077                         edr_events, proxy, firewall, ad_events   176   90.0                   23053                      74.0                        93.5
  RB-03 ALR-20261018-005343                         idp_signins, idp_audit, vpn, ad_events  2068 1215.1                  311076                      98.1                        99.5
  RB-04 ALR-20261016-004962                       dlp, sql_audit, edr_events, email, proxy   333  164.7                   42157                      85.8                        96.4
  RB-05 ALR-20261018-00547

**Observed:** The first table covers every alert centre, without an entity filter. The second is one reproducible representative per runbook, with exact raw alert entities; it is not an alias-complete retrieval benchmark. Sources are mapped from runbook checks; rule → runbook comes from each alert and the catalogue.

**Design interpretation (data/context, agent/prompt):** An 8,000-token prompt needs a fixed evidence budget and space for policy, instructions and output. Aggregate repeated events, retain state transitions, retrieve linked events directly, and offer bounded follow-ups. `bytes/4` is an estimate, not a measured tokenizer count; the conservative byte ceiling is intentionally stricter. Actual model tokenisation and answer quality remain unverified.

## 6. Severity

How many alerts name a known machine, only a user, or nothing internal? Apply the policy matrix. Where do the policy, runbooks, deck and sample ticket disagree?

Affects: data/severity

In [7]:
severity_order=["informational","low","medium","high","critical"]
criticality_order=["Baja","Media","Alta","Crítica"]
matrix=pd.DataFrame([
 ["informational","informational","low","low"],
 ["informational","low","low","medium"],
 ["low","medium","medium","high"],
 ["medium","high","high","critical"],
 ["high","critical","critical","critical"]],index=severity_order,columns=criticality_order)
show("Policy matrix (before bounded adjustment)",matrix.reset_index(names="rule_severity"))
asset_alias={str(a[k]).casefold():a for a in assets.to_dict("records") for k in ["hostname","ip"] if a[k]}
user_alias={str(u[k]).casefold():u for u in users.to_dict("records") for k in ["upn","username"] if u[k]}
# Explicit analytical assumption, not a claim that the CMDB provides these relations.
service_fallback={"RB-01":"Alta","RB-03":"Crítica","RB-05":"Crítica","RB-07":"Alta","RB-08":"Alta"}
severity_rows=[]
for a in alerts.to_dict("records"):
    vals={v.casefold() for v in entity_values(a)}
    named_assets={asset_alias[v]["hostname"]:asset_alias[v] for v in vals if v in asset_alias}
    named_users={user_alias[v]["upn"]:user_alias[v] for v in vals if v in user_alias}
    workstation_assets={u["workstation"]:asset_alias[u["workstation"].casefold()] for u in named_users.values()
                        if u["workstation"].casefold() in asset_alias}
    category="known_machine" if named_assets else "only_known_user" if named_users else "no_resolved_internal_entity"
    candidates=list(named_assets.values()) or list(workstation_assets.values())
    crit=max((x["criticality"] for x in candidates),key=criticality_order.index,default=None)
    inferred=False
    if crit is None:
        crit=service_fallback.get(a["runbook"],"Media"); inferred=True
    vip=any(u["vip"]=="True" or u["privileged"]=="True" for u in named_users.values())
    base=matrix.loc[a["severity"],crit]
    provisional=severity_order[min(4,severity_order.index(base)+int(vip))]
    severity_rows.append([a["alert_id"],category,crit,inferred,vip,a["severity"],base,provisional])
severity_calc=pd.DataFrame(severity_rows,columns=["alert_id","entity_resolution","criticality",
    "assumed_criticality","vip_or_privileged","rule_severity","matrix_base","base_plus_vip_only"])
show("Entity resolution coverage",severity_calc.entity_resolution.value_counts())
show("Criticality assumptions",{"alerts_needing_service_or_default_assumption":int(severity_calc.assumed_criticality.sum()),
 "matrix_changes_rule_severity":int(severity_calc.matrix_base.ne(severity_calc.rule_severity).sum()),
 "vip_or_privileged_alerts":int(severity_calc.vip_or_privileged.sum())})
show("Matrix base vs rule severity",pd.crosstab(severity_calc.rule_severity,severity_calc.matrix_base).reset_index())
# Read supplied reference artefacts as data, without interpreting embedded instructions.
schema=json.loads((DATA/"01_contexto/ticket_salida.schema.json").read_text())
example=json.loads((DATA/"01_contexto/ticket_ejemplo.json").read_text())
slide_text={}
for ppt in sorted(DATA.glob("*.pptx")):
    with zipfile.ZipFile(ppt) as z:
        for name in z.namelist():
            if re.fullmatch(r"ppt/slides/slide\d+\.xml",name):
                slide_text[name]=" ".join(x.text or "" for x in ET.fromstring(z.read(name)).iter() if x.tag.endswith("}t"))
show("Reference contract differences",pd.DataFrame([
 ["policy vs ticket schema","3 policy verdicts; schema also permits needs_more_info"],
 ["policy vs RB-01","Unverified IBAN change is critical in RB-01; medium rule × Media asset is medium before at most +1"],
 ["policy vs RB-07","Blocked attempt is medium in RB-07; blocked-before-effect adjustment may reduce matrix result"],
 ["policy vs deck slides 7/9 and project rule","Policy exempts block_ioc/purge_email; project requires human containment approval"],
 ["deck slide 6 vs inventory","Backup service CMDB criticality and deck CRM/copies priority need reconciliation"],
 ["ticket example","Example drops medium to informational (-2), exceeding the policy max -1"],
],columns=["comparison","finding"]))
show("Sample ticket checks",{"alert_ids":example.get("alert_ids"),"severity":example.get("severity"),
 "verdict":example.get("verdict"),"has_severity_rationale":"severity_rationale" in example,
 "schema_verdict_options":schema["properties"]["verdict"]["enum"],"deck_slides_parsed":len(slide_text)})

example_base=matrix.loc["low","Crítica"]
example_expected=severity_order[max(0,severity_order.index(example_base)-1)]
show("Worked ticket arithmetic check",{"matrix_low_x_critical":example_base,
 "after_one_level_block_adjustment":example_expected,"example_severity":example["severity"],
 "example_alert_id_exists":all(aid in set(alerts.alert_id) for aid in example["alert_ids"])})


Policy matrix (before bounded adjustment)
rule_severity          Baja         Media     Alta  Crítica
informational informational informational      low      low
          low informational           low      low   medium
       medium           low        medium   medium     high
         high        medium          high     high critical
     critical          high      critical critical critical



Entity resolution coverage
                      value  count
no_resolved_internal_entity   4026
              known_machine   3818
            only_known_user   3217

Criticality assumptions
{'alerts_needing_service_or_default_assumption': 4037, 'matrix_changes_rule_severity': 3570, 'vip_or_privileged_alerts': 44}

Matrix base vs rule severity
rule_severity  critical  high  low  medium
     critical        12     1    0       0
         high       152   622    0       1
          low         0     0 2856    3416
       medium         0     0    0    4001

Reference contract differences
                                comparison                                                                                    finding
                   policy vs ticket schema                                     3 policy verdicts; schema also permits needs_more_info
                           policy vs RB-01 Unverified IBAN change is critical in RB-01; medium rule × Media asset is medium before...
    

**Observed:** The matrix is applied to all alerts under the displayed entity-resolution assumptions. The table is a **base-severity calculation**, not final triage. Users have no criticality field. VIP/privilege is a possible +1 adjustment, never multiple increases. Unknown external entities are not automatically low risk.

**Design interpretation (data/severity):** Code the matrix, provenance and a single bounded adjustment. Record unresolved identity/service criticality. Confirm execution and subsequent activity before raising/lowering severity; apply the authorised-activity override only with validated scope and time. Clarify runbook conflicts with a person and retain the rule, matrix base, adjustment and final result separately. A rule's critical rating does not establish malice.

The supplied example assigns `informational` after describing a `medium` matrix base and a −1 adjustment: one step would be `low`. Its alert ID is a placeholder absent from the dataset. Treat this as a concrete reference inconsistency, not an instruction to copy the result.

## 7. Time

What time zone are alerts, logs and the change calendar in? Check against real alerts that match approved changes, including across the clock change.

Affects: data/context

In [8]:
calendar = change_times(changes)
show("Timestamp formats",pd.DataFrame([
 ["alerts",len(alerts),int(alerts.created_at.astype(str).str.endswith("Z").sum()),int(alerts.created_at_utc.isna().sum())],
 *[[s,len(d),int(d.timestamp.astype(str).str.endswith("Z").sum()),int(d.timestamp_utc.isna().sum())] for s,d in logs.items()],
 ["change calendar start/end",2*len(changes),0,0]],columns=["source","values","explicit_Z","invalid_event_times"]))
# Calendar has no offset: test Madrid local against a fixed UTC+2 interpretation.
scan=alerts[alerts.rule_id.eq("UC-NET-003") & alerts.entities.map(lambda e:
    any(v=="NESSUS01" for _,v in leaves(e)))].copy()
clock_rows=[]
for date in ["2026-10-07","2026-10-14","2026-10-21","2026-10-28"]:
    local=scan.created_at_utc.dt.tz_convert("Europe/Madrid")
    part=scan[local.dt.strftime("%Y-%m-%d").eq(date)]
    cid=calendar.loc[calendar.inicio.str.startswith(date),"id"].iloc[0]
    c=calendar.set_index("id").loc[cid]
    fixed_start=pd.Timestamp(c.inicio).tz_localize("UTC")-pd.Timedelta(hours=2)
    fixed_end=pd.Timestamp(c.fin).tz_localize("UTC")-pd.Timedelta(hours=2)
    clock_rows.append([cid,len(part),part.created_at_utc.min().isoformat(),
       part.created_at_utc.dt.tz_convert("Europe/Madrid").min().isoformat(),
       int(part.created_at_utc.between(c.inicio_utc,c.fin_utc).sum()),
       int(part.created_at_utc.between(fixed_start,fixed_end).sum())])
show("Scanner alerts matching real change windows, before/after DST",pd.DataFrame(clock_rows,
    columns=["change_id","scanner_alerts","first_UTC","first_Madrid","inside_Madrid_window","inside_fixed_plus2_window"]))
show("Madrid clock change",pd.DataFrame([
 [t,t.tz_convert("Europe/Madrid").isoformat()] for t in pd.to_datetime(
     ["2026-10-25T00:30:00Z","2026-10-25T01:30:00Z"],utc=True)
],columns=["UTC","Madrid_local"]))
# Independent approved maintenance example.
maint=alerts[alerts.rule_id.eq("UC-EDR-011") & alerts.evidence.map(lambda e:
    "CHG-2026-1080" in approval_refs(e))]
show("Pre-change maintenance cross-check",{"change":"CHG-2026-1080","alerts":len(maint),
 "inside_Madrid_window":int(change_window(maint.created_at_utc,"CHG-2026-1080",calendar).sum())})


Timestamp formats
                   source  values  explicit_Z  invalid_event_times
                   alerts   11061       11061                    0
                ad_events     526         526                    0
               ai_gateway    8817        8817                    0
              cicd_builds     559         559                    0
               cloudtrail    6496        6496                    0
                      dlp    1042        1042                    0
                      dns  178221      178221                    0
             ecom_account    7661        7661                    0
                ecom_auth   44064       44064                    0
               edr_events  112492      112492                    0
                    email   56797       56797                    0
             email_clicks   16249       16249                    0
                 firewall   24648       24648                    0
                idp_audit     762         76

**Observed:** All event timestamps checked above have explicit UTC (`Z`). Calendar start/end values have no offset. Scanner events move by one UTC hour after the clock change and match the Madrid wall-time schedule; this supports interpreting the calendar as `Europe/Madrid`. The README's fixed UTC+2 statement does not hold throughout this period.

**Design interpretation (data/context):** Preserve UTC for comparisons and convert Madrid calendar wall times using zoneinfo, not a fixed offset. Raise on ambiguous or nonexistent local calendar times. The examples support this interpretation, but no formal calendar timezone declaration was supplied. The displayed events also fall within the wider fixed-offset windows, so window inclusion alone cannot distinguish those hypotheses; the repeated local start-time pattern supplies the additional support.

## 8. Authorised activity

Which alerts match approved changes, approved flows, HR events or company rules?

Affects: agent/prompt, data/context

In [9]:
# These are reproducible contextual matches, not automatically assigned verdicts.
context_matches=[]
authorised_ids=set()
def context_match(name, mask, constraint, validated=False):
    ids=set(alerts.loc[mask,"alert_id"])
    context_matches.append([name,len(ids),constraint,"scoped match" if validated else "lead"])
    if validated: authorised_ids.update(ids)
    return ids
has_value=lambda v: alerts.entities.map(lambda e:any(x==v for _,x in leaves(e)))
scanner_mask=alerts.alert_id.isin(scan.alert_id)
scanner_window=pd.Series(False,index=alerts.index)
for cid in ["CHG-2026-1049","CHG-2026-1050","CHG-2026-1051","CHG-2026-1052"]:
    scanner_window |= change_window(alerts.created_at_utc,cid,calendar)
# Check referenced firewall rows also remain outside OT.
def no_ot_refs(a):
    rows=[logs[s].loc[i] for eid in a.related_events for s,i in event_index.get(eid,[]) if s=="firewall"]
    return bool(rows) and all(r.zone_dst!="LOGISTICS-OT" for r in rows)
scan_scoped=alerts.apply(no_ot_refs,axis=1)
context_match("authorised scanner",scanner_mask & scanner_window & scan_scoped,
              "NESSUS01 + Wednesday change window + referenced destinations outside OT",True)
sccm=alerts.rule_id.eq("UC-EDR-002") & alerts.evidence.map(lambda e:
    "CcmExec" in str(e.get("parent","")) and e.get("user")=="SYSTEM")
sccm_window=change_window(alerts.created_at_utc,"CHG-2026-1031",calendar)
local_hour=alerts.created_at_utc.dt.tz_convert("Europe/Madrid").dt.hour
context_match("SCCM inventory",sccm & sccm_window & local_hour.between(8,9),
              "approved period + stated local 08:00-10:00 + parent/account; command review still needed")
context_match("SCCM pattern outside stated daily window",sccm & sccm_window & ~local_hour.between(8,9),
              "same pattern; verify schedule rather than broadly allowlisting")
pentest=alerts.rule_id.eq("UC-WEB-001") & alerts.entities.map(lambda e:
    bool(set(e.get("ips",[])) & {"192.0.2.10","192.0.2.11","192.0.2.12"}))
context_match("pentest",pentest & change_window(alerts.created_at_utc,"CHG-2026-1042",calendar),
              "approved IP + window; request path/scope must also be reviewed")
ps=alerts.rule_id.eq("UC-NET-004") & has_value("JUMP01") & alerts.evidence.map(lambda e:
    e.get("dst") in {"ERP-APP01","HR-APP01","PRINT01","CRM-APP01"})
context_match("PsExec patching",ps & change_window(alerts.created_at_utc,"CHG-2026-1057",calendar),
              "JUMP01 + listed target + approved window; identity/method needs review")
flow=alerts.rule_id.eq("UC-DLP-002") & alerts.evidence.map(lambda e:e.get("recipient")=="asesoria@gestoria-pons.example")
context_match("payroll flow",flow & change_window(alerts.created_at_utc,"FLUJO-0007",calendar),
              "recipient + active flow; verify payroll sender/contents")
context_match("OT allowed company source",alerts.rule_id.eq("UC-OT-001") & (has_value("ENG-WS01")|has_value("WMS01")),
              "company authorised source; never execute OT containment",True)
context_match("OT maintenance",alerts.rule_id.eq("UC-OT-001") & has_value("JUMP01") &
              change_window(alerts.created_at_utc,"CHG-2026-1063",calendar),
              "source/window; confirm temporary firewall rule and PLC scope")
context_match("directory group change",alerts.rule_id.eq("UC-IDP-008") & has_value("JUMP01") &
              change_window(alerts.created_at_utc,"CHG-2026-1078",calendar),
              "source/window/change reference; verify named group/member")
context_match("EDR agent upgrade",alerts.alert_id.isin(maint.alert_id) &
              change_window(alerts.created_at_utc,"CHG-2026-1080",calendar),
              "server window + calendar ID; verify short stop and restart")
context_match("WMS supplier exception",alerts.rule_id.eq("UC-VPN-001") & has_value("ext.logisoft@tarvia.example") &
              has_value("198.18.240.10") & change_window(alerts.created_at_utc,"CHG-2026-1071",calendar),
              "supplier + corporate IP + exceptional window; inspect WMS-only access")
show("Authorised-activity/context candidates",pd.DataFrame(context_matches,
    columns=["context","alerts","constraints_checked_or_remaining","status"]))
hr_summary=[]
for kind, rows in hr.groupby("tipo"):
    matched=alerts.entities.map(lambda e:any(str(v).casefold() in set(rows.usuario.str.casefold()) for _,v in leaves(e)))
    hr_summary.append([kind,len(rows),int(matched.sum())])
show("HR identity joins (all dates; not proof of travel or wrongdoing)",pd.DataFrame(hr_summary,
    columns=["HR_type","HR_rows","alerts_for_same_identity"]))
calendar_ids=set(changes.id)
claimed_refs=collections.Counter(ref for e in alerts.evidence for ref in approval_refs(e))
show("Approval references in alert evidence",pd.DataFrame([
 [ref,count,ref in calendar_ids] for ref,count in sorted(claimed_refs.items())],
 columns=["reference","alert_occurrences","exists_in_calendar"]))

# Apply the policy's authorised-activity severity cap only to the two scoped
# analytical matches above. This is a candidate calculation for human review.
scoped_base=severity_calc[severity_calc.alert_id.isin(authorised_ids)].copy()
scoped_base["candidate_authorised_severity"]=scoped_base.matrix_base.map(
    lambda value:"informational" if value=="informational" else "low")
show("Scoped-match policy override candidates",pd.crosstab(
    scoped_base.matrix_base,scoped_base.candidate_authorised_severity).reset_index())


Authorised-activity/context candidates
                                 context  alerts                                                         constraints_checked_or_remaining       status
                      authorised scanner      19                  NESSUS01 + Wednesday change window + referenced destinations outside OT scoped match
                          SCCM inventory    1331 approved period + stated local 08:00-10:00 + parent/account; command review still needed         lead
SCCM pattern outside stated daily window      18                           same pattern; verify schedule rather than broadly allowlisting         lead
                                 pentest      20                           approved IP + window; request path/scope must also be reviewed         lead
                         PsExec patching       4                   JUMP01 + listed target + approved window; identity/method needs review         lead
                            payroll flow       2       

HR identity joins (all dates; not proof of travel or wrongdoing)
                   HR_type  HR_rows  alerts_for_same_identity
                      alta        3                        14
baja_voluntaria_comunicada        3                        10
             cambio_puesto        2                         5
      teletrabajo_temporal        2                        10
                vacaciones        3                         6
                     viaje        4                        14

Approval references in alert evidence
    reference  alert_occurrences  exists_in_calendar
CHG-2026-1071                  1                True
CHG-2026-1078                  1                True
CHG-2026-1080                  6                True
     SEC-2210                  3               False
     SEC-4471                 12               False

Scoped-match policy override candidates
matrix_base  low
       high    9
     medium   19



**Observed:** Counts above distinguish scoped matches from incomplete contextual leads. Calendar IDs and dates alone do not validate scope. HR joins deliberately do not infer travel destinations or date ranges from an effective date. Accepted MFA exceptions do not authorise a service-account compromise; approved software also needs a matching process/signature/context.

**Design interpretation (agent/prompt, data/context):** Treat authorisation as a conjunction of identity, source, destination, action and time. Keep unmatched constraints in the ticket. Do not close solely because an alert mentions a change, a known vendor, a corporate tool or HR activity. These counts are not ground-truth benign labels.

## 9. Attacker-written text

Which free-text fields contain content addressed to analysts or automated systems, hidden formatting, or claims of approval? Do claimed approvals exist in the change calendar? What are the limits of this search?

Affects: agent/prompt, agent/triage

In [10]:
# Heuristic screen of every string leaf, including nested structures.
# Categories overlap. A hit is a candidate for review, not a proven injection.
screens={
 "instruction_override":re.compile(r"(?:ignore|disregard).{0,55}(?:previous|prior|instruction|system)|ignora.{0,55}(?:instrucciones|anteriores|sistema)",re.I|re.S),
 "analyst_or_agent_directive":re.compile(r"(?:analyst|analista|automated|automatizado|agente|SOC).{0,100}(?:cerr|close|benign|false.positive|falso.positivo|aprob|instrucc)|(?:do not escalate|no escalar|cierra.{0,25}alerta)",re.I|re.S),
 "approval_reference":re.compile(r"\b(?:SEC-\d+|CHG-\d{4}-\d+|FLUJO-\d+)\b"),
 "hidden_formatting":re.compile(r"[\u200b-\u200f\u202a-\u202e\u2066-\u2069\ufeff]")}
screens["hidden_HTML"]=re.compile(r"<!--|display\s*:\s*none|visibility\s*:\s*hidden|font-size\s*:\s*0|aria-hidden",re.I)
screens["customer_correction_language"]=re.compile(r"(?:ignora|olvida).{0,60}(?:anterior|pedido|direcci)",re.I|re.S)
screens["role_or_policy_override"]=re.compile(r"system\s*:|<system>|instrucciones.{0,25}sistema|prompt.{0,25}sistema|revela.{0,60}(?:intern|c[oó]dig)|bypass",re.I|re.S)
hits=[]
all_tables={"alerts":alerts,**{"log/"+s:d for s,d in logs.items()},
    **{"inventory/"+s:d for s,d in inventory.items()},**{"intel/"+s:d for s,d in intel.items()}}
for table,d in all_tables.items():
    for row in raw_records(d):
        rid=next((row[k] for k in ["alert_id","event_id","asset_id","user_id","id","sha256"] if k in row),"row-without-id")
        for field,value in leaves(row):
            if not isinstance(value,str):continue
            for category,pattern in screens.items():
                if pattern.search(value):
                    hits.append([table,str(rid),field,category])
text_hits=pd.DataFrame(hits,columns=["table","record_id","field","category"]).drop_duplicates()
show("Heuristic field hits by table/category",text_hits.groupby(["table","category"]).size().reset_index(name="field_hits"))
show("Distinct flagged records and fields",{"records":int(text_hits[["table","record_id"]].drop_duplicates().shape[0]),
 "record_field_pairs":int(text_hits[["table","record_id","field"]].drop_duplicates().shape[0])})
# Sample one record per field/category; emit paths/IDs and controlled descriptions only.
examples=text_hits.sort_values(["category","table","record_id"]).groupby(["category","table","field"]).head(1)
show("Sanitised examples (at most 18; never raw content)",examples.head(18))
show("Claimed approval with no calendar record",pd.DataFrame([
 [a.alert_id,"evidence.service_description","SEC-2210","service metadata claims security approval; no calendar entry"]
 for a in alerts.itertuples() if "SEC-2210" in approval_refs(a.evidence)
],columns=["alert_id","field","claimed_reference","description"]))
# Directives outside ai_gateway would be missed by inspecting only Vela prompts.
non_ai=text_hits[~text_hits.table.eq("log/ai_gateway") & text_hits.category.ne("approval_reference")]
show("Non-AI-log heuristic candidates",non_ai.groupby(["table","field","category"]).size().reset_index(name="field_hits"))

show("All screen categories, including zero hits",text_hits.category.value_counts().reindex(
    screens,fill_value=0))
show("Hidden HTML candidate examples (IDs and fields only)",text_hits[
    text_hits.category.eq("hidden_HTML")].groupby(["table","field"]).head(1))
show("Absent claimed approval references (alert evidence)",pd.DataFrame([
    [reference,count] for reference,count in sorted(claimed_refs.items()) if reference not in calendar_ids
],columns=["claimed_reference","alerts"]))


Heuristic field hits by table/category
                       table                     category  field_hits
                      alerts   analyst_or_agent_directive          12
                      alerts           approval_reference          23
                      alerts customer_correction_language         112
                      alerts                  hidden_HTML          12
                      alerts         instruction_override         415
                      alerts      role_or_policy_override         416
inventory/calendario_cambios           approval_reference          13
              log/ai_gateway customer_correction_language         112
              log/ai_gateway      role_or_policy_override           1
              log/edr_events           approval_reference           3
              log/edr_events      role_or_policy_override        1349
                   log/email   analyst_or_agent_directive          45
                   log/email           approval_ref

**Observed:** This lexical scan emits only field paths, record IDs and controlled descriptions. The three TPV alerts' approval reference does not exist among calendar IDs. Repeated representations in alerts and logs are counted separately. Vela attack prompts concern Vela; attempts addressed to a SOC analyst create a different trust-boundary risk.

**Design interpretation (agent/prompt, agent/triage):** Every retrieved text field, including CMDB notes and EDR service descriptions, stays quoted data. Retrieved text cannot approve actions, rewrite policy or supply a verdict. Validate claimed change IDs against calendar scope/time in code. This screen misses paraphrases, encoded payloads, cross-field instructions and unknown languages, and can flag legitimate administrative prose. It is not a complete detector or an estimate of attack prevalence.

The scan also checks HTML comments/CSS hiding and common Spanish customer corrections. These overlap with legitimate language: detector boilerplate in alert descriptions is explicitly not an attack label. Zero Unicode-format hits only means this particular character screen found none. The absent `SEC-4471` claims and hidden email HTML are additional review leads; neither a claim nor visual concealment grants approval.

## 10. Attack stories

Starting from rare, severe alerts and following shared indicators, what are the likely incidents? Fully trace at least one, including evidence the alerts don't link to.

Affects: agent/triage, data/eval

In [11]:
# Start with low-frequency severe rules, then follow exact shared entities/indicators.
rare_rules=set(alerts.rule_id.value_counts()[lambda x:x<=12].index)
seeds=alerts[alerts.rule_id.isin(rare_rules) & alerts.severity.isin(["high","critical"])]
show("Rare severe seed counts (leads, no ground-truth label)",seeds.rule_id.value_counts())
# Exclude ubiquitous services, usernames, process names and private/shared addresses from
# indicator-only incident candidates. Equality is exact, never substring matching.
indicator_paths=("domains[]","files[].sha256","urls[]")
indicator_to_alerts=collections.defaultdict(set)
for a in alerts.itertuples():
    for field,value in leaves(a.entities):
        if field in indicator_paths and value: indicator_to_alerts[(field,str(value))].add(a.alert_id)
shared_groups={tuple(sorted(ids)) for ids in indicator_to_alerts.values() if len(ids)>1 and ids & set(seeds.alert_id)}
show("Exact indicator groups intersecting rare severe seeds",pd.DataFrame([
 [len(ids),len(set(alerts.loc[alerts.alert_id.isin(ids),"rule_id"])),min(ids),max(ids)]
 for ids in sorted(shared_groups,key=lambda ids:(-len(ids),ids))[:10]
],columns=["alerts","rules","first_alert_id","last_alert_id"]))
# Canonical case: trace events rather than relying on alert arrival order.
story_ids=["ALR-20261007-011785","ALR-20261007-011786","ALR-20261007-011787",
           "ALR-20261007-011788","ALR-20261008-011789"]
story=alerts[alerts.alert_id.isin(story_ids)]
story_refs={eid for ids in story.related_events for eid in ids}
account="laura.ferrer2@tarvia.example"
start,end=pd.Timestamp("2026-10-07T00:00Z"),pd.Timestamp("2026-10-09T00:00Z")
# Follow exact user -> session -> audit and exact message_id -> mail/click.
phish=alerts.set_index("alert_id").loc[story_ids[0]]
message_id=phish.evidence["message_id"]
message_rows=logs["email"][logs["email"].message_id.eq(message_id)]
clicks=logs["email_clicks"][logs["email_clicks"].message_id.eq(message_id)]
session_id=alerts.set_index("alert_id").loc[story_ids[2]].evidence["session_id"]
signins=logs["idp_signins"][logs["idp_signins"].session_id.eq(session_id)]
audits=logs["idp_audit"][logs["idp_audit"].actor_upn.eq(account) &
    logs["idp_audit"].timestamp_utc.between(start,end)]
fraud=logs["email"][logs["email"].event_id.isin(
    alerts.set_index("alert_id").loc[story_ids[-1]].related_events)]
transitions=[]
for label,d in [("phishing mail delivered",message_rows),("link click allowed",clicks),
                ("phishing site GET/POST",logs["proxy"][logs["proxy"].event_id.isin(phish.related_events)]),
                ("same session, changed user agent",signins),("mailbox audit",audits),
                ("bank-change message delivered",fraud)]:
    for r in d.itertuples():
        transitions.append([r.timestamp_utc,label,r.event_id,r.event_id in story_refs])
timeline=pd.DataFrame(transitions,columns=["timestamp_UTC","stage","event_id","linked_by_story_alert"])
show("Canonical story: event stage totals",timeline.groupby(["stage","linked_by_story_alert"]).agg(
    events=("event_id","size"),first_UTC=("timestamp_UTC","min"),last_UTC=("timestamp_UTC","max")).reset_index())
show("Canonical story: chronological turning points",timeline.sort_values("timestamp_UTC").groupby("stage").head(2).sort_values("timestamp_UTC"))
show("Canonical story: audit operations not linked by any story alert",audits[
    ~audits.event_id.isin(story_refs)].operation.value_counts())
show("Canonical story: arrival vs occurrence",pd.DataFrame([
 [a.alert_id,a.rule_id,a.created_at_utc,min(logs[s].loc[i,"timestamp_utc"]
   for e in a.related_events for s,i in event_index.get(e,[]))] for a in story.itertuples()
],columns=["alert_id","rule","alert_created_UTC","earliest_referenced_event_UTC"]))
show("Canonical story: observed scope",{"alerts":len(story),"unique_linked_events":len(story_refs),
 "unlinked_mail_access_events":int((audits.operation.eq("MailItemsAccessed") & ~audits.event_id.isin(story_refs)).sum()),
 "shared_session_distinct_origins":int(signins.src_ip.nunique()),
 "shared_session_distinct_user_agents":int(signins.user_agent.nunique()),
 "fraud_amount_claimed_in_alert":alerts.set_index("alert_id").loc[story_ids[-1]].evidence.get("amount_mentioned"),
 "bank_message_logged":len(fraud)})
# Other chains are explicitly leads. Association is based on exact host or identity;
# it cannot prove the same intruder or that an action succeeded.
lead_specs=[
 ("credential theft to ransomware",["WS-BCN-1770","FS01"],["UC-EDR-001","UC-NET-001","UC-EDR-004","UC-NET-003","UC-NET-004","UC-EDR-007"]),
 ("vendor VPN / DNS / directory / agent tampering",["ext.logisoft@tarvia.example","WMS01","PRINT01"],["UC-VPN-001","UC-NET-006","UC-IDP-008","UC-EDR-011"]),
 ("TPV memory reads and later network",["POS-T07-01","POS-T07-02","POS-T07-03"],["UC-EDR-012","UC-NET-001"]),
 ("possible dependency / cloud chain",["CI-RUNNER01"],["UC-DEV-001"]),
 ("Vela session disclosure",[],["UC-AI-002"]),
]
lead_rows=[]
for name,entities,rules in lead_specs:
    mask=alerts.rule_id.isin(rules)
    if entities:mask &= alerts.entities.map(lambda e:any(v in entities for _,v in leaves(e)))
    part=alerts[mask]
    lead_rows.append([name,len(part),part.created_at_utc.min(),part.created_at_utc.max(),
                      part.alert_id.iloc[0] if len(part) else "none","lead; not fully traced"])
show("Other candidate chains",pd.DataFrame(lead_rows,columns=["hypothesis","alerts","first_UTC","last_UTC","seed_alert_id","status"]))
# Local TIP coverage only. Unknown means absent from this feed, not benign.
feed_values=set(intel["ti_feed"].value)
alert_indicators={v for a in alerts.itertuples() for field,v in leaves(a.entities)
                  if field in {"domains[]","ips[]","files[].sha256"} and isinstance(v,str)}
show("Local intelligence coverage",{"distinct_alert_indicators":len(alert_indicators),
 "present_in_TIP":len(alert_indicators & feed_values),"absent_from_TIP":len(alert_indicators-feed_values)})

show("Raw-log support and limits for the canonical case",pd.DataFrame([
 ["sign-in session",len(signins),"one source IP, two user agents; session changes client, not logged source"],
 ["proxy POST",int((logs["proxy"].event_id.isin(phish.related_events) & logs["proxy"].method.eq("POST")).sum()),
  "submission request logged; payload and actual credential contents unavailable"],
 ["mailbox access",len(audits[audits.operation.eq("MailItemsAccessed")]),"operations logged; not independent proof of who controlled the client"],
],columns=["check","events","interpretation_limit"]))
# Explicit final-policy candidate for this traced case: use all affected identities'
# workstations and mailbox service criticality, then one +1 for confirmed mailbox access.
# Fraud runbook asks for critical; the reason for escalation is independent of severity.
canonical_base=matrix.loc[max(story.severity,key=severity_order.index),"Alta"]
canonical_candidate=severity_order[min(4,severity_order.index(canonical_base)+1)]
show("Canonical grouped ticket policy candidate",{"service_criticality_assumption":"Alta (mail service)",
 "highest_rule_severity":"high","matrix_base":canonical_base,"single_access_adjustment":"+1",
 "candidate_final_severity":canonical_candidate,"escalate":True,
 "basis":"session anomaly, mailbox persistence/access, attempted financial fraud; human review required"})


Rare severe seed counts (leads, no ground-truth label)
     value  count
UC-WEB-003     12
 UC-OT-001     10
UC-EDR-011      7
UC-NET-004      5
UC-EML-003      4
UC-EDR-012      3
UC-NET-002      2
UC-IDP-004      2
UC-IDP-006      2
UC-IDP-008      2
UC-EDR-006      1
UC-IDP-005      1
 UC-AI-002      1
UC-EDR-004      1
UC-EDR-007      1
UC-IDP-003      1
UC-CLD-001      1
UC-CLD-002      1
UC-CLD-004      1

Exact indicator groups intersecting rare severe seeds
 alerts  rules      first_alert_id       last_alert_id
      3      1 ALR-20261020-011897 ALR-20261020-011899
      3      1 ALR-20261020-011911 ALR-20261020-011913
      2      2 ALR-20261010-011797 ALR-20261018-011896

Canonical story: event stage totals
                           stage  linked_by_story_alert  events                 first_UTC                  last_UTC
   bank-change message delivered                   True       1 2026-10-08 08:15:00+00:00 2026-10-08 08:15:00+00:00
              link click allowed         

Other candidate chains
                                    hypothesis  alerts                     first_UTC                      last_UTC       seed_alert_id                 status
                credential theft to ransomware       7 2026-10-09 09:31:10.000+00:00 2026-10-27 17:05:53.568+00:00 ALR-20261009-011791 lead; not fully traced
vendor VPN / DNS / directory / agent tampering      24 2026-10-09 16:10:34.000+00:00 2026-10-29 03:22:44.000+00:00 ALR-20261009-002236 lead; not fully traced
            TPV memory reads and later network       4 2026-10-20 08:04:48.000+00:00 2026-10-21 07:33:30.000+00:00 ALR-20261020-011897 lead; not fully traced
             possible dependency / cloud chain      11 2026-10-12 16:17:55.312+00:00 2026-10-28 09:26:16.191+00:00 ALR-20261012-003200 lead; not fully traced
                       Vela session disclosure       1 2026-10-08 20:20:56.000+00:00 2026-10-08 20:20:56.000+00:00 ALR-20261008-011823 lead; not fully traced



Local intelligence coverage
{'distinct_alert_indicators': 4921, 'present_in_TIP': 31, 'absent_from_TIP': 4890}

Raw-log support and limits for the canonical case
          check  events                                                          interpretation_limit
sign-in session       2     one source IP, two user agents; session changes client, not logged source
     proxy POST       1 submission request logged; payload and actual credential contents unavailable
 mailbox access      40         operations logged; not independent proof of who controlled the client

Canonical grouped ticket policy candidate
{'service_criticality_assumption': 'Alta (mail service)', 'highest_rule_severity': 'high', 'matrix_base': 'high', 'single_access_adjustment': '+1', 'candidate_final_severity': 'critical', 'escalate': True, 'basis': 'session anomaly, mailbox persistence/access, attempted financial fraud; human review required'}



**Observed, traced case:** Mail delivery → click and proxy GET/POST → two user agents using the same session from one logged source → inbox rule → mailbox reads → an internal bank-change message. Exact message ID, session ID, identity and referenced event IDs connect the chain. The bank-change amount is a claim in the alert/mail evidence; settlement or actual financial loss is not demonstrated. Forty mailbox-access events are outside the five alerts' explicit links. The phishing alert arrives after several later attack stages, so sorting alerts by creation time would misorder the attack.

**Interpretation (agent/triage, data/eval):** Escalate the combined case for credential/session compromise, persistence and attempted fraud. A click alone would not prove credential theft; the correlated session and mailbox activity provide the extra evidence. Group related alerts and retain evidence provenance. The logs do not establish a change of IP within the suspicious session, the exact theft mechanism, or the submitted credential contents. The wider candidate chains remain leads: common hosts/tools or indicators do not establish a single adversary. TIP absence is not a clean bill of health. No live reputation lookup, threat attribution, actual exfiltration volume, payment settlement, or containment outcome was verified.

## 11. Test set

Does the 150 alert sample cover what we found? Which alert IDs should be added?

Affects: data/eval

In [12]:
sample_ids=set(sample.alert_id)
coverage=alerts.groupby("rule_id").size().to_frame("full_alerts").join(
    sample.groupby("rule_id").size().rename("sample_alerts")).fillna(0).astype(int)
show("Sample rule coverage",coverage.reset_index())
show("Sample severity coverage",pd.crosstab(sample.rule_id,sample.severity).sum())
show("Runbook and source coverage",{"full_runbooks":alerts.runbook.nunique(),"sample_runbooks":sample.runbook.nunique(),
    "full_products":alerts.source_product.nunique(),"sample_products":sample.source_product.nunique(),
    "observed_rules_missing_from_sample":int(coverage.sample_alerts.eq(0).sum()),
    "catalog_rules_without_any_alert":len(set(catalog.rule_id)-set(alerts.rule_id))})
# Add a bounded, reviewable set of behavioural checks, not just one alert per rule.
test_candidates={aid:"canonical session/mail/fraud chain" for aid in story_ids}
for rule in coverage[coverage.sample_alerts.eq(0)].index:
    test_candidates[alerts.loc[alerts.rule_id.eq(rule),"alert_id"].iloc[0]]="observed rule absent from sample"
for aid,reason in [
 ("ALR-20261020-011897","service-description approval claim absent from calendar"),
 ("ALR-20261024-011916","approved EDR stop vs unapproved tampering"),
 ("ALR-20261018-011896","unapproved EDR tampering counterpart"),
 ("ALR-20261028-011864","post-DST scanner window"),
 ("ALR-20261007-011850","pre-DST scanner window"),
 ("ALR-20261016-011922","approved vendor out-of-hours exception"),
 ("ALR-20261011-011877","new vendor source and out-of-hours lead"),
 ("ALR-20261023-011915","approved privileged group change"),
 ("ALR-20261008-011823","Vela multi-event disclosure investigation"),
 ("ALR-20261013-011901","build secrets access vs ordinary dependency scripts"),
]:
    if aid in set(alerts.alert_id):test_candidates[aid]=reason
recommended=pd.DataFrame([[aid,reason,aid in sample_ids] for aid,reason in test_candidates.items()],
    columns=["alert_id","test_purpose","already_in_sample"])
show("Behavioural test selection",recommended)
show("Additions required",{"new_alert_ids":int((~recommended.already_in_sample).sum()),
 "canonical_chain_alerts_in_sample":len(sample_ids & set(story_ids))})
show("Held-out evaluation precaution",{"alert_id":"ALR-20261016-011821",
 "in_development_sample":"ALR-20261016-011821" in sample_ids,
 "use":"reserve from prompt tuning; connected cloud story is not an independent holdout"})


Sample rule coverage
   rule_id  full_alerts  sample_alerts
 UC-AI-001          415              6
 UC-AI-002            1              0
UC-CLD-001            1              0
UC-CLD-002            1              0
UC-CLD-003           50              1
UC-CLD-004            1              1
UC-DEV-001           11              0
UC-DLP-001           59              1
UC-DLP-002            3              0
UC-EDR-001          532              4
UC-EDR-002         1349             19
UC-EDR-003          411              4
UC-EDR-004            1              0
UC-EDR-005            1              0
UC-EDR-006            1              0
UC-EDR-007            1              0
UC-EDR-008            1              0
UC-EDR-011            7              0
UC-EDR-012            3              0
UC-EML-001         2295             34
UC-EML-002            1              0
UC-EML-003            4              1
UC-EML-004            1              0
UC-IDP-001          901             11
UC-I

**Observed:** Rule coverage alone does not establish coverage of authorisation exceptions, prompt injection, event ordering, DST or same-name identities. The explicit list above selects concrete cases without modifying the sample file.

**Design interpretation (data/eval):** Keep full incident groups together when splitting train/development/test, otherwise linked alerts leak the answer. Ask a person to label expected verdict, severity, escalation, cited evidence and required approvals. Measure retrieval completeness and group quality as well as per-alert accuracy. The official live set is unavailable; generalisation to unseen alerts has not been tested. The proposed held-out cloud alert is not independently held out from related chain evidence, so do not claim a clean held-out incident score.

## 12. Mock API

On a copy outside data/, what does the API enforce compared with section 4 of the policy? What must our own code enforce?

Affects: data/api-client, agent/triage

In [13]:
from analysis.api_probe import probe_api
api_results,api_summary=probe_api()
show("Actual requests against an unchanged temporary mock API copy",api_results)
show("Probe summary",api_summary)
show("Enforcement required in application code",pd.DataFrame([
 ["approval identity","Authenticated human; reject model-created approvals and arbitrary approver strings"],
 ["approval scope","Bind canonical target/action/ticket, expiry and single use; store audit evidence"],
 ["asset classes","N2 + service owner for servers; N2 + business for TPV/web"],
 ["OT","Resolve hostname/case/IP aliases; never perform SOC containment on OT"],
 ["IOC blocking","Human approval under project rule; dual-use service block needs N2"],
 ["input bounds","Positive bounded limits; known targets; source allowlist; explicit incomplete results"],
 ["search results","Canonical exact entities, pagination by time, timestamps, provenance, deduplication"],
],columns=["area","required_control"]))


/tmp/tarvia-analysis-venv/lib/python3.14/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


Actual requests against an unchanged temporary mock API copy
                                     probe  HTTP_status
                     isolate without token          403
             token for wrong action/target          403
                        unsupported action          400
                block_ioc without approval          200
              purge_email without approval          200
                     unknown host accepted          200
critical server, arbitrary single approver          200
                       same token replayed          200
                         TPV, one approver          200
                         OT exact hostname          403
                        OT lowercase alias          200
                               OT IP alias          200
              OT IP block without approval          200
       known dual-use service global block          200
                     unknown user accepted          200
                        unknown log source 

**Observed:** Tests use FastAPI's in-process client with no listening port. The API source is copied unchanged to a temporary directory; read-only dataset inputs are linked, and the containment journal is written only beside that copy and deleted on exit. Synthetic approval fixtures test the mock contract; they are not operational approvals.

The API validates supported action names, token action/target equality and an exact-case OT hostname for `isolate_host`. It accepts arbitrary approver text, reusable tokens, unknown targets, a single approver for a critical server/TPV, OT aliases, and IOC blocks without approval. Its `truncated` flag means the result reached the limit, even when no additional record exists. These are measured mock behaviours, not authorisation to use them in the application.

**Design interpretation (data/api-client, agent/triage):** The application must enforce policy and the stricter project rule that a person approves every containment action. Keep approval and execution tools outside LLM authority. No changes to other branches are made here. Missing checks in the mock need a wrapper, not a prompt promise.

**Verification limits:** All twelve sections are executed. Other attack stories remain leads, lexical injection recall is unknown, token counts are estimates, authorisation candidates require the remaining scope checks, and the dataset provides no independent alert-level ground truth. Real indicator reputation, real-world effects, live demo alerts and model quality are deliberately untested.